# E3: Attention Fusion + Random Emoji Embeddings

**Experiment**: Text-conditioned attention fusion with randomly-initialized emoji embeddings.

**Architecture**: Frozen BERT (768-d) + random emoji embedding (32-d) → attention fusion → concat (800-d) → MLP → 3 classes

**Controlled-experiment contract**:
- Same train/validation/test splits as E0/E1/E2
- Text branch receives ONLY `text_without_emoji`
- Emoji branch receives ONLY `emoji_list`
- Emoji embeddings are RANDOMLY INITIALIZED (no pretrained)

**Runtime**: GPU (T4 recommended)

## 1. Setup

In [ ]:
# Clone repository
!rm -rf /content/SentimentAnalysis
!git clone https://github.com/Chetnapadhi/SentimentAnalysis.git /content/SentimentAnalysis
%cd /content/SentimentAnalysis
!pwd

In [ ]:
# Verify GPU
import torch

print('PyTorch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())

if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
    props = torch.cuda.get_device_properties(0)
    print(f'VRAM: {props.total_memory / 1024**3:.1f} GB')

In [ ]:
# Install dependencies
!pip install -q -r requirements.txt
!python -c "import torch, transformers, pandas, sklearn; print('Environment OK')"

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## 2. Build Canonical Data Pipeline

Same data preparation as E0/E1/E2 — downloads StockTwits, recovers labels, builds canonical splits.

In [ ]:
%cd /content/SentimentAnalysis

# Step 1: Inspect datasets (downloads from HuggingFace)
!python -m src.data.inspect_datasets

# Step 2: Recover StockTwits labels from source files
!python -m src.data.stocktwits_adapter

# Step 3: Build final deduplicated dataset
!python -m src.data.build_final_dataset

# Step 4: Create canonical JSONL files
!python -m src.data.preprocessing

In [ ]:
# Verify canonical data
import os
import pandas as pd

base = 'data/processed/canonical'

for split in ['train', 'validation', 'test']:
    path = f'{base}/final_{split}.jsonl'
    exists = os.path.exists(path)
    if exists:
        df = pd.read_json(path, lines=True)
        print(f'{split}: {len(df)} rows, columns={list(df.columns)}')
    else:
        print(f'{split}: MISSING!')

assert os.path.exists(f'{base}/final_train.jsonl'), 'Canonical data not found!'

## 3. Train E3

In [ ]:
%cd /content/SentimentAnalysis

# Run E3 training + evaluation
!RUN_E3=1 python run_e3.py

## 4. Check Results

In [ ]:
# List all E3 artifacts
!find results/E3 -maxdepth 2 -type f | sort

In [ ]:
# View E3 metrics
import json

with open('results/E3/metrics.json') as f:
    metrics = json.load(f)

print('=' * 60)
print('E3 OFFICIAL RESULTS')
print('=' * 60)
print(f"Accuracy:        {metrics['accuracy']:.6f}")
print(f"Macro Precision: {metrics['macro_precision']:.6f}")
print(f"Macro Recall:    {metrics['macro_recall']:.6f}")
print(f"Macro F1:        {metrics['macro_f1']:.6f}")
print(f"Best Epoch:      {metrics['best_epoch']}")
print(f"Best Val F1:     {metrics['best_val_macro_f1']:.6f}")

In [ ]:
# View confusion matrix image
from IPython.display import Image, display

display(Image('results/E3/confusion_matrix.png'))
display(Image('results/E3/training_history.png'))

## 5. Backup to Google Drive

In [ ]:
import shutil
import os

drive_dir = '/content/drive/MyDrive/SentimentAnalysis/results/E3'
os.makedirs(drive_dir, exist_ok=True)

!cp -r results/E3/* {drive_dir}/

print('E3 results backed up to Drive.')
!find {drive_dir} -maxdepth 2 -type f | sort